# O4

CODE:

In [1]:
# Importing the data
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import jaccard_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.svm import LinearSVC
import os

Ant_dat = "antenna_fault.csv"

# Load the data into a DataFrame
df = pd.read_csv(Ant_dat)


# print(df.head()) #Prints the first five rows to get an idea of how the dataset looks
# print(df.info()) #Prints some infomation about the data and their type (String/float)
# print(df.describe()) #Prints count, mean, standard deviation, and the quatiles for each feature
# df.hist(bins=50, figsize=(20,15))
# plt.show()

This repport aims to build a model with the abilty to both determine whether or not, there is a fault in the bluetooth and/or wifi antenna system, and if there is, what type of fault is present. 
For the O4 project we have chosen to investigate the reliabilty of flexible and wearable antennas. With the rise of wearable technoligies like smartwatches there is a massive demand for reliable hardware cabable of suriving a bit of sweat or a knock. Our goal will be to develop a machine learning model able to predict imminent fault cases and be a part of the maintenance and detection system. The machine learning model should be a classifier able to ascertain whether or not the wifi and or bluetooth is failing and also classify which kind of fault is present. To train our model we have chosen [This data set from kaggle.com](https://www.kaggle.com/datasets/amineipad/antenna-performance-and-fault-detection-dataset?fbclid=IwY2xjawQffuJleHRuA2FlbQIxMQBzcnRjBmFwcF9pZAEwAAEeJ2UxEK4SsTF78Us3f-ALRqmDz68CFkv2aszLsnn0NTB3mq0-Kfri-Ghet-M_aem_khEFRjyx4IXRwn4GQIPcEw), it should work well due to its comprehensive feature list and a sample size of 2689, which is much more than is stated on the website, but alas is the true value of samples. The features include Length, Width, Height, Permittivity, Conductivity, and epsilon_r. Alongside these physical traits, the dataset captures critical radio frequency (RF) performance indicators like Return Loss (S11), Voltage Standing Wave Ratio (VSWR), Gain, Efficiency, and Bandwidth. From the description of the dataset: "Crucially, the dataset features explicit classifications for various environmental and physical fault conditions—such as Humidity/Sweat, Bending, Cracks, and Rupture/Coupure—and tracks their impact on the operational status of both WiFi and BT functionalities. It is an excellent resource for predictive maintenance, anomaly detection, and robust wearable antenna design using machine learning." Furthermore we as a group hope this project will enable us to obtain a deeper understanding of antenna perfomance and the parameters related to this. The data in this set was "collected through a series of controlled simulations and empirical measurements that emulate real-world conditions for wearable antennas. Various physical deformations and environmental stressors were applied to the antenna models, including bending at different radii, introducing structural cracks or ruptures, and simulating changes in dielectric properties due to humidity or sweat." So we should have in mind that the data probably represents cleaner versions of the faults and is limited to these kinds of fault, not entirely covering the real world in which the wearable antennas would exist. First, lets prepare the data for our model.

In [2]:
# Removing the wifi status and Bluetooth status columns to avoid leaking the answers
# Drop columns at index 14 and 15
ds = df.drop(df.columns[[14, 15]], axis=1)
print(ds.head())

   Length  Width  Height  Permittivity  Conductivity  Bend   Feed    S11  \
0   46.72  38.62    1.46          1.51      12210.65  0.56 -10.27  -7.09   
1   40.32  36.17    1.30          1.61       9589.73  0.48 -17.33  -7.92   
2   43.84  35.89    1.17          1.53       4067.44  0.99 -11.95 -18.97   
3   43.77  39.46    1.26          1.88      13052.58  0.67 -10.07 -12.89   
4   46.48  39.66    1.30          2.44      12682.11  1.08 -17.57 -22.94   

   VSWR  Gain  Efficiency  Bandwidth       WiFi Fault        BT Fault  \
0  3.41  5.09       62.26     102.74           Cracks          Cracks   
1  2.47  5.17       66.61      96.06  Rupture_Coupure        No_Fault   
2  3.20  5.07       83.42     150.03          Bending  Strong_Flexion   
3  1.72  3.83       42.22      54.23      Body_Effect         Coupure   
4  1.44  3.48       77.39      93.00   Strong_Flexion        No_Fault   

   epsilon_r  
0       3.90  
1       3.03  
2       3.65  
3       3.51  
4       2.06  


In [3]:
# Splitting the data into features and target categories
target_cols = ['WiFi Fault', 'BT Fault']
y = ds[target_cols].copy()
X = ds.drop(target_cols, axis=1)

# Splitting into train and test sets, using stratify parameter for the y to ensure uniform distribution of faults
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y 
)
print("X_train")
print(X_train.head())
print("X_test")
print(X_test.head())
print("y_train")
print(y_train.head())
print("y_test")
print(y_test.head())


X_train
      Length  Width  Height  Permittivity  Conductivity  Bend   Feed    S11  \
691    48.65  33.06    1.42          2.23       5418.60  0.42 -21.84 -11.52   
845    45.45  38.88    1.45          1.65       3111.05  0.53 -17.70 -24.38   
2682   45.61  30.12    1.39          1.84       3548.36  0.41 -20.20  -4.03   
1557   44.69  32.21    0.84          1.92       4435.72  0.11 -21.19  -3.70   
277    46.27  39.87    1.10          2.04      12281.38  0.26 -15.50  -6.19   

      VSWR  Gain  Efficiency  Bandwidth  epsilon_r  
691   1.75  5.46       42.19      69.46       3.10  
845   3.43  2.61       88.56     140.02       1.99  
2682  2.06  1.02       51.55     153.63       1.68  
1557  1.48  2.52       70.16      65.94       3.72  
277   3.64  2.69       77.88     104.07       3.05  
X_test
      Length  Width  Height  Permittivity  Conductivity  Bend   Feed    S11  \
1044   48.95  38.75    1.10          2.16      14932.32  0.58 -12.05  -3.49   
2560   40.10  34.26    1.24       

In [5]:
# Initiating the classifier
forest_clf = RandomForestClassifier(random_state=42)
forest_clf.fit(X_train, y_train)
y_pred = forest_clf.predict(X_test)

print(y_test.shape)
print(y_pred.shape)
from sklearn.metrics import hamming_loss



y_test_np = np.array(y_test)
y_pred_np = np.array(y_pred)

# 2. Flatten both to 1D (538x2 becomes 1056)
y_test_flat = y_test_np.ravel()
y_pred_flat = y_pred_np.ravel()


score = accuracy_score(y_test_flat, y_pred_flat)
print(score)

# Get the loss
loss = hamming_loss(y_test_flat, y_pred_flat)

# Convert to score
hamming_score = 1 - loss

print(f"Hamming Score: {hamming_score:.4f}")

j_score = jaccard_score(y_test_flat, y_pred_flat, average='macro')

print(f"Jaccard Score: {j_score:.4f}")

(538, 2)
(538, 2)
0.879182156133829
Hamming Score: 0.8792
Jaccard Score: 0.8001
